# 1 — Leakage Diagnosis & Dataset Cleaning

Diagnoses duplicate feature-signature leakage in the raw industrial textile-chemical dataset and produces the cleaned dataset (`data/thesis_clean.csv`) used by `02_benchmark.ipynb`.

**Paper:** *Duplicate Feature Signatures Inflate Tabular Model Accuracy: A Leakage-Aware Re-Benchmark on Industrial Quality Data* — submitted to ICCIT 2026.

In [ ]:
# Setup: clone the repo so relative paths work (skip if already running inside it, e.g. locally)
import os
if not os.path.exists('data/thesis_dataset.csv'):
    !git clone -q https://github.com/Al-AminKhan043/leakage-aware-tabular-benchmark.git
    %cd leakage-aware-tabular-benchmark


## Load the raw dataset

In [ ]:
import os
if not os.path.exists("data/thesis_dataset.csv"):
    raise FileNotFoundError(
        "data/thesis_dataset.csv not found.\n"
        "Raw data isn't included in this repo (see data/README.md) -- "
        "place your own copy at data/thesis_dataset.csv before running this cell."
    )

In [ ]:
import pandas as pd

df = pd.read_csv("data/thesis_dataset.csv")

print(df.shape)
df.head()

## Missing values

In [ ]:
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())

## Cardinality per column

In [ ]:
df.nunique().sort_values()

## Duplicate rows: with vs. without the `Date` column

In [ ]:
with_date = df.duplicated().sum()
feature_cols = [c for c in df.columns if c != "Date"]
without_date = df.duplicated(subset=feature_cols).sum()

print("Total rows:                  ", len(df))
print("Duplicate rows WITH Date:    ", with_date)
print("Duplicate rows WITHOUT Date: ", without_date)
print("Duplication rate without Date: {:.1%}".format(without_date / len(df)))

## `Boiling Point`: does each chemical map to exactly one value?

In [ ]:
# does each chemical map to exactly one boiling point?
print(df.groupby("Chemical Name")["Boiling Point"].nunique())

print("\nThe actual values:")
print(df.groupby("Chemical Name")["Boiling Point"].unique())

## Impute `Boiling Point` from each chemical's known value

In [ ]:
# fill missing boiling points using each chemical's known value
df["Boiling Point"] = df.groupby("Chemical Name")["Boiling Point"].transform(
    lambda x: x.fillna(x.iloc[0] if x.notna().any() else x)
)

print("Missing after lookup fill:", df["Boiling Point"].isnull().sum())

## Sentinel value for chemicals with no boiling point

Sodium Alginate decomposes before boiling, so it has no true boiling point. We use a sentinel (`-1`) rather than dropping it, so tree-based models can treat "no boiling point" as its own informative category.

In [ ]:
# Sodium Alginate has no boiling point (decomposes before boiling).
# Use a sentinel so the model can treat it as its own category, not a real temperature.
df["Boiling Point"] = df["Boiling Point"].fillna(-1)

print("Missing values remaining:", df["Boiling Point"].isnull().sum())
print(df["Boiling Point"].value_counts())

## Build feature signatures and count unique ones

A **signature** is the exact combination of feature values in a row (everything except `Date` and the target `Quality`). This is the core diagnostic: **9,310 rows collapse onto only 235 unique signatures.**

In [ ]:
# define predictive features = everything except Date and the target
exclude = ["Date", "Quality"]
feature_cols = [c for c in df.columns if c not in exclude]

# a "signature" = the exact combination of feature values in a row
df["signature"] = df[feature_cols].astype(str).agg("|".join, axis=1)

print("Total rows:               ", len(df))
print("Unique feature signatures:", df["signature"].nunique())
print("\nRows per signature (top 5):")
print(df["signature"].value_counts().head())

## Save the cleaned dataset

In [ ]:
df.to_csv("data/thesis_clean.csv", index=False)
print("Saved. Shape:", df.shape)

## Sanity check: reload and confirm the signature count

In [ ]:
import pandas as pd
df = pd.read_csv("data/thesis_clean.csv")
print(df.shape)
df.head()

In [ ]:
print("Unique signatures:", df["signature"].nunique())